# Challenge I: let's build a game!

Can you find a hidden word by chasing its meaning? If the answer is **library**, you might try **book**, then **school**. Your game will tell you whether you’re getting hotter or colder and offer clues along the way. Crack the answer quickly and in fewer guesses to earn more points.

Here’s the twist: you’ll build the whole thing yourself, including the word embeddings. You’ll teach randomly initialized vectors using WordNet definitions and word relationships, then turn what they learn into a playable guessing game. No pretrained vectors or language-model API doing the guessing for you!

**Your word playground:** you can guess any WordNet 3.0 lemma made entirely of letters a–z, including rare words. Multiword, hyphenated, and accented entries sit this round out. Words found only in definitions can help train the model, but don’t count as guesses. Your hidden answers will come from a curated list of familiar nouns.

**Bring your toolkit:** you’ll use the tensors, gradients, and training loops from previous exercises, plus Python classes and dictionaries. Tackle the tasks in order. After each implementation, run the example cell to see your work in action. Explore the outputs: does your game’s idea of “close” agree with yours?

| Part | Tasks | What you’ll build |
|---|---|---|
| Dictionary and examples | 1–6 | Your vocabulary and training/evaluation pairs |
| Embedding model and learning | 7–14 | Your embedding model and training loop |
| Geometry and persistence | 15–19 | Neighbors, plots, and a saved model |
| Game mechanics | 20–26 | Clues, scoring, feedback, and a playable game |
| Investigations | 27–30 | Experiments with feedback leakage, evaluation, subwords, and senses |


### Read then Run: environment and WordNet

Use Python 3.10+ with `torch`, `nltk`, and `numpy`. If needed, install them in a separate cell with `%pip install torch nltk numpy matplotlib`, then restart the kernel. Matplotlib is optional. CPU is supported; CUDA is selected when available. The download is cached under `semantic_data/corpora/wordnet.zip`.

WordNet groups word senses into **synsets** with definitions and lexical relations. A word may belong to several synsets. [WordNet](https://wordnet.princeton.edu/), [NLTK interface](https://www.nltk.org/howto/wordnet.html), and [WordNet license](https://wordnet.princeton.edu/license-and-commercial-use). Keep its license with any redistributed dictionary data. Some source entries contain offensive or dated language; the default target list is curated rather than sampled from every entry.


In [ ]:
import copy
import hashlib
import json
import math
import random
import re
import time
import urllib.request
import zipfile
from collections import Counter, defaultdict
from dataclasses import dataclass, field
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
import nltk
from nltk.corpus import wordnet as wn

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(min(4, torch.get_num_threads()))
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATA_DIR = Path("semantic_data")
ARCHIVE = DATA_DIR / "corpora" / "wordnet.zip"
ARCHIVE.parent.mkdir(parents=True, exist_ok=True)
nltk.data.path.insert(0, str(DATA_DIR.resolve()))
if not ARCHIVE.exists():
    url = "https://raw.githubusercontent.com/nltk/nltk_data/gh-pages/packages/corpora/wordnet.zip"
    temporary = ARCHIVE.with_suffix(".partial")
    try:
        with urllib.request.urlopen(url, timeout=90) as response:
            temporary.write_bytes(response.read())
        with zipfile.ZipFile(temporary) as archive:
            assert "wordnet/data.noun" in archive.namelist()
        temporary.replace(ARCHIVE)
    finally:
        temporary.unlink(missing_ok=True)
assert wn.get_version() == "3.0", "Use WordNet 3.0 for reproducible dictionary contents"
DATA_HASH = hashlib.sha256(ARCHIVE.read_bytes()).hexdigest()
print("WordNet:", wn.get_version(), "| device:", DEVICE)
print("Dictionary SHA-256:", DATA_HASH)

# Supplied building blocks: focus on the learning logic in the tasks below.
def embedding_table(rows, dimensions, scale=0.1):
    table = torch.nn.Embedding(rows, dimensions, sparse=True)
    torch.nn.init.normal_(table.weight, std=scale)
    return table

def make_optimizer(model, lr):
    # This optimizer supports the sparse gradients produced by embedding lookups.
    return torch.optim.SparseAdam(model.parameters(), lr=lr)

def stable_softplus(values):
    # Computes log(1 + exp(values)) without overflow.
    return F.softplus(values)


### Part I: turn a dictionary into training data

A list of spellings supplies identifiers, not semantic supervision. Definitions and lexical relations supply evidence: **library** and **book** can share contexts about reading, collections, and documents. The model must learn that structure from repeated examples.


## Task 1: Choose a consistent word format

Implement `normalize_word` for guesses and `tokenize_definition` for prose. Accept guesses only when their trimmed, lowercase spelling contains letters a–z; return `None` otherwise. Definition tokens are lowercase alphabetic runs.


In [ ]:
def normalize_word(text):
    ### Your Solution starts here ###
    pass
    ### Your Solution ends here ###

def tokenize_definition(text):
    ### Your Solution starts here ###
    pass
    ### Your Solution ends here ###


In [ ]:
# Try it: inspect the output
for raw in [" Library ", "ice cream", "co-op", "book!"]:
    print(f"{raw!r} → {normalize_word(raw)!r}")
print("Definition tokens:", tokenize_definition("A building with books and reading rooms."))


## Task 2: Build the dictionary vocabulary

Implement `dictionary_words` to collect every eligible WordNet spelling in a reproducible order. Include rare words, but exclude multiword and punctuated entries.


In [ ]:
def dictionary_words(lemma_names):
    ### Your Solution starts here ###
    pass
    ### Your Solution ends here ###

headwords = dictionary_words(wn.all_lemma_names())
headword_set = set(headwords)
print("Accepted dictionary spellings:", len(headwords))


In [ ]:
# Try it: inspect the output
sample_entries = ["Book", "book", "ice_cream", "stadium", "co-op"]
print("Dictionary entries:", sample_entries)
print("Accepted spellings:", dictionary_words(sample_entries))
print(f"Full dictionary: {len(headwords):,} accepted spellings")


## Task 3: Find semantic evidence

Implement `collect_contexts` to associate each headword with evidence from definitions, synonyms, and direct hypernyms across its senses. Normalize definition words morphologically, remove stop words and self-pairs, and retain at most 64 distinct contexts, favoring lexical relations. Every headword needs at least one context.


In [ ]:
STOP_WORDS = set("a an the and or of to in on at by for from with as is are was were be been being this that these those it its into than which who whose having have has had not no any some such one two used especially usually something someone make made consisting belonging relating".split())

def collect_contexts(synsets, allowed, cap=64):
    ### Your Solution starts here ###
    pass
    ### Your Solution ends here ###

contexts = collect_contexts(wn.all_synsets(), headword_set)
print("Context edges:", sum(map(len, contexts.values())))
print("Some library contexts:", contexts["library"][:16])


In [ ]:
# Try it: inspect the output
for word in ["library", "book", "stadium"]:
    print(f"{word}: {contexts[word][:12]}")


## Task 4: Assign vocabulary IDs

Implement `make_vocabularies` for dictionary words and context words. Dictionary words must share the same IDs in both vocabularies; additional context tokens must remain separate from accepted guesses.


In [ ]:
def make_vocabularies(headwords, contexts):
    ### Your Solution starts here ###
    pass
    ### Your Solution ends here ###

word_to_id, context_to_id = make_vocabularies(headwords, contexts)
id_to_word = list(word_to_id)
id_to_context = list(context_to_id)
print("Headword rows:", len(word_to_id), "| context rows:", len(context_to_id))


In [ ]:
# Try it: inspect the output
for word in ["book", "library", "stadium"]:
    print(f"{word}: headword ID={word_to_id[word]}, context ID={context_to_id[word]}")
print("Context-only tokens:", id_to_context[len(headwords):len(headwords) + 10])


## Task 5: Separate training and evaluation

Implement `split_contexts` with a reproducible split of approximately 80% training, 10% validation, and 10% test edges. A pair and its reverse must stay together, held-out pairs must be unique, and every headword must retain training evidence. Reserve test edges for the final audit.


In [ ]:
def pair_key(a, b):
    return tuple(sorted((a, b)))

def edge_bucket(a, b, seed):
    encoded = json.dumps([seed, *pair_key(a, b)], separators=(",", ":")).encode()
    return int.from_bytes(hashlib.sha256(encoded).digest()[:4], "little") % 10

def split_contexts(contexts, seed=SEED):
    ### Your Solution starts here ###
    pass
    ### Your Solution ends here ###

train_contexts, validation_edges, test_edges = split_contexts(contexts)
print("Training headwords:", len(train_contexts), "| validation/test edges:", len(validation_edges), len(test_edges))


In [ ]:
# Try it: inspect the output
print("Library training contexts:", train_contexts["library"][:10])
print("Validation pairs:", validation_edges[:5])
print("Test pairs reserved for the final audit:", len(test_edges))


## Task 6: Represent training examples compactly

Implement `pack_contexts` to store variable-length context rows in a flat ID tensor with offsets and lengths. Implement `encode_edges` to convert word pairs to ID tensors. The representation should support sampling without a dense vocabulary-by-vocabulary matrix.


In [ ]:
def pack_contexts(headwords, rows, context_to_id):
    ### Your Solution starts here ###
    pass
    ### Your Solution ends here ###

def encode_edges(edges, word_to_id, context_to_id):
    ### Your Solution starts here ###
    pass
    ### Your Solution ends here ###

flat_contexts, offsets, lengths = pack_contexts(headwords, train_contexts, context_to_id)
val_heads, val_contexts = encode_edges(validation_edges, word_to_id, context_to_id)
print("Packed training edges:", len(flat_contexts))


In [ ]:
# Try it: inspect the output
for word in ["book", "library"]:
    row = word_to_id[word]
    ids = flat_contexts[offsets[row]:offsets[row + 1]][:8]
    print(f"{word}: stored IDs {ids.tolist()}")
    print("Decoded contexts:", [id_to_context[i] for i in ids])


### Part II: train word embeddings

Learn one shared embedding table from positive dictionary pairs and sampled negative contexts. This adapts negative-sampling learning to dictionary evidence rather than running text. Every full epoch visits every headword; this guarantees coverage, not semantic quality.

For a batch of B examples, headword and positive IDs have shape `(B,)`, negative IDs `(B, K)`, and vectors have dimension D. The model returns logits of shape `(B,)` and `(B, K)`.


## Task 7: Define the noise distribution

Implement `noise_distribution` using training-context frequencies raised to 0.75. Return a normalized CPU float32 distribution, with zero probability for unobserved contexts.


In [ ]:
def noise_distribution(flat, context_count, exponent=0.75):
    ### Your Solution starts here ###
    pass
    ### Your Solution ends here ###

noise = noise_distribution(flat_contexts, len(context_to_id))


In [ ]:
# Try it: inspect the output
probabilities = noise.tolist()
top_ids = sorted(range(len(probabilities)), key=lambda i: -probabilities[i])[:8]
print("Most likely noise contexts:")
for token_id in top_ids:
    print(f"{id_to_context[token_id]:20s} {float(noise[token_id]):.4%}")


## Task 8: Draw negative examples

Implement `sample_negatives` to draw K noise contexts per positive pair reproducibly using a local Python random generator. Exclude the positive context and any supplied headword ID; repeated negatives are allowed. Reject requests with no available alternative.


In [ ]:
def sample_negatives(positive_ids, noise, k, generator, forbidden_ids=None):
    ### Your Solution starts here ###
    pass
    ### Your Solution ends here ###


In [ ]:
# Try it: inspect the output
example_heads = torch.tensor([word_to_id["library"]])
example_positive = torch.tensor([context_to_id[train_contexts["library"][0]]])
example_negatives = sample_negatives(example_positive, noise, 5,
                                    random.Random(7), forbidden_ids=example_heads)
print("Headword: library")
print("Positive context:", id_to_context[example_positive[0]])
print("Sampled negatives:", [id_to_context[i] for i in example_negatives[0]])


## Task 9: Build the embedding model

Implement `DictionaryEmbeddings` with one shared sparse embedding table for headwords and contexts. Initialize small random vectors and return positive and negative dot-product logits, shifted by log(K). The model should support batches of positive pairs and K negatives per pair.


In [ ]:
class DictionaryEmbeddings(torch.nn.Module):
    def __init__(self, headword_count, context_count, dim=96):
        super().__init__()
        ### Your Solution starts here ###
        pass
        ### Your Solution ends here ###

    def forward(self, heads, positives, negatives):
        ### Your Solution starts here ###
        pass
        ### Your Solution ends here ###


In [ ]:
# Try it: inspect the output
example_model = DictionaryEmbeddings(2, 4, dim=8)
positive_logits, negative_logits = example_model(
    torch.tensor([0, 1]), torch.tensor([2, 3]), torch.tensor([[1, 3], [0, 2]]))
print("Positive-pair logits:", positive_logits.detach().tolist())
print("Negative-pair logits:", negative_logits.detach().tolist())
print("These scores come from an untrained model.")


## Task 10: Define the learning objective

Implement `negative_sampling_loss`: reward positive pairs and penalize sampled negative pairs. For each example use $-\log\sigma(s^+) - \sum_k \log\sigma(-s^-_k)$, then average over examples. Handle extreme logits stably and reject empty or incompatible inputs.


In [ ]:
def negative_sampling_loss(positive, negative):
    ### Your Solution starts here ###
    pass
    ### Your Solution ends here ###


In [ ]:
# Try it: inspect the output
for label, positive, negative in [
    ("Undecided", [0.], [[0., 0.]]),
    ("Correctly separated", [3.], [[-3., -3.]]),
    ("Reversed predictions", [-3.], [[3., 3.]])]:
    value = negative_sampling_loss(torch.tensor(positive), torch.tensor(negative))
    print(f"{label}: loss={float(value):.3f}")


## Task 11: Perform a learning step

Implement `embedding_step` to update the model from one batch using the supplied `make_optimizer` helper. Return the loss before the update. Negative sampling remains outside this function.


In [ ]:
def embedding_step(model, optimizer, heads, positives, negatives):
    ### Your Solution starts here ###
    pass
    ### Your Solution ends here ###


In [ ]:
# Try it: inspect the output
example_model = DictionaryEmbeddings(2, 4, dim=8)
example_optimizer = make_optimizer(example_model, lr=0.03)
example_batch = (torch.tensor([0, 1]), torch.tensor([2, 3]), torch.tensor([[1, 3], [0, 2]]))
with torch.no_grad():
    before_logits = example_model(*example_batch)[0].clone()
step_loss = embedding_step(example_model, example_optimizer, *example_batch)
with torch.no_grad():
    after_logits = example_model(*example_batch)[0]
print("Loss before update:", round(step_loss, 3))
print("Positive logits before:", before_logits.tolist())
print("Positive logits after: ", after_logits.tolist())


## Task 12: Cover the full dictionary each epoch

Implement `epoch_batches` so each headword appears exactly `draws_per_word` times per epoch, in shuffled order, paired with randomly selected training contexts. Sampling must be reproducible and include the final partial batch.


In [ ]:
def epoch_batches(flat, offsets, lengths, batch_size, draws_per_word, generator):
    ### Your Solution starts here ###
    pass
    ### Your Solution ends here ###


In [ ]:
# Try it: inspect the output
example_batches = epoch_batches(flat_contexts, offsets, lengths, batch_size=4,
                                draws_per_word=1, generator=random.Random(7))
example_heads, example_contexts = next(example_batches)
print("One sampled batch:")
for head, context in zip(example_heads, example_contexts):
    print(f"{id_to_word[head]} → {id_to_context[context]}")


## Task 13: Evaluate held-out pairs

Implement `evaluate_edges` to report average loss and how often a positive score exceeds a fixed negative score, counting ties as half. Evaluate in bounded batches without updating the model.


In [ ]:
def evaluate_edges(model, heads, positives, negatives, batch_size=1024):
    ### Your Solution starts here ###
    pass
    ### Your Solution ends here ###

val_negatives = sample_negatives(val_contexts, noise, 8, random.Random(2026), forbidden_ids=val_heads)


In [ ]:
# Try it: inspect the output
example_model = DictionaryEmbeddings(len(headwords), len(context_to_id), dim=8)
example_metrics = evaluate_edges(example_model, val_heads[:100], val_contexts[:100], val_negatives[:100])
print("Untrained model on 100 held-out edges:")
print(json.dumps(example_metrics, indent=2))


## Task 14: Train and select a checkpoint

Implement `train_embeddings` for reproducible full-dictionary training. Record loss, validation discrimination, and headword coverage each epoch. Return the checkpoint with the highest validation discrimination, breaking ties by lower validation loss, together with the training history.


In [ ]:
CONFIG = {"dim": 96, "epochs": 18, "batch_size": 2048, "draws_per_word": 4,
          "negatives": 8, "lr": 0.025, "seed": SEED}

def train_embeddings(config, model_factory=DictionaryEmbeddings):
    ### Your Solution starts here ###
    pass
    ### Your Solution ends here ###

embedding_model, training_history = train_embeddings(CONFIG)
best_row = training_history[0]
for row in training_history[1:]:
    if row["pairwise"] > best_row["pairwise"]:
        best_row = row
    elif row["pairwise"] == best_row["pairwise"] and row["val_loss"] < best_row["val_loss"]:
        best_row = row
selected_epoch = best_row["epoch"]
print("Selected epoch:", selected_epoch)


In [ ]:
# Try it: inspect the output
print("First epoch:", training_history[0])
print("Selected epoch:", training_history[selected_epoch - 1])
print("Restored checkpoint:", evaluate_edges(embedding_model, val_heads, val_contexts, val_negatives))


### Part III: turn learned vectors into a semantic index

Training scores dictionary headwords against context words in a shared embedding table. The game compares two **headwords**, using cosine similarity between their learned rows. This is a new use of the representation and requires its own inspection: a low training loss alone does not certify good game feedback.


## Task 15: Prepare vectors for cosine similarity

Implement `unit_rows` and `inference_vectors` to export centered, unit-length headword vectors for the game. Reject nonfinite values and rows without a direction. Inference vectors must be detached from training and stored on CPU.


In [ ]:
def unit_rows(matrix):
    ### Your Solution starts here ###
    pass
    ### Your Solution ends here ###

def inference_vectors(model):
    ### Your Solution starts here ###
    pass
    ### Your Solution ends here ###

with torch.no_grad():
    game_vectors = inference_vectors(embedding_model)


In [ ]:
# Try it: inspect the output
example_vectors = torch.tensor([[3., 4.], [0., 2.]])
print("Original vectors:\n", example_vectors)
print("Unit vectors:\n", unit_rows(example_vectors))
print("Library inference vector (first 8 coordinates):", game_vectors[word_to_id["library"], :8].tolist())


## Task 16: Retrieve semantic neighbors

Implement `SemanticIndex` to look up cosine scores and nearest neighbors from a vocabulary and its vectors. Preserve the word-to-row mapping, exclude the query from its neighbors, and resolve ties consistently.


In [ ]:
class SemanticIndex:
    def __init__(self, words, vectors):
        ### Your Solution starts here ###
        pass
        ### Your Solution ends here ###

    def scores(self, word):
        ### Your Solution starts here ###
        pass
        ### Your Solution ends here ###

    def neighbors(self, word, k=10):
        ### Your Solution starts here ###
        pass
        ### Your Solution ends here ###

semantic_index = SemanticIndex(headwords, game_vectors)
print("Learned neighbors of library:", semantic_index.neighbors("library", 10))


In [ ]:
# Try it: inspect the output
for word in ["library", "apple", "river"]:
    print(f"\nNeighbors of {word}:")
    for neighbor, cosine in semantic_index.neighbors(word, 5):
        print(f"  {neighbor:20s} {cosine:.3f}")


## Task 17: Inspect semantic quality

Implement `triple_accuracy` to compare related and unrelated words using the supplied triples, counting ties as half. Compare learned vectors with random vectors and inspect the library/book/stadium example. Report the actual results, even when they are unintuitive.


In [ ]:
DEVELOPMENT_TRIPLES = [
    ("library", "book", "stadium"), ("cat", "dog", "airplane"),
    ("apple", "pear", "hammer"), ("doctor", "nurse", "volcano"),
    ("ocean", "sea", "pencil"), ("piano", "violin", "potato"),
    ("bicycle", "vehicle", "cloud"), ("river", "stream", "chair"),
    ("teacher", "school", "banana"), ("forest", "tree", "computer"),
    ("bread", "food", "planet"), ("winter", "summer", "screwdriver")]

def triple_accuracy(index, triples):
    ### Your Solution starts here ###
    pass
    ### Your Solution ends here ###

random_vectors = torch.randn(game_vectors.shape, generator=torch.Generator().manual_seed(7))
random_index = SemanticIndex(headwords, random_vectors - random_vectors.mean(dim=0, keepdim=True))
learned_accuracy, triple_rows = triple_accuracy(semantic_index, DEVELOPMENT_TRIPLES)
random_accuracy, _ = triple_accuracy(random_index, DEVELOPMENT_TRIPLES)
print("Development triple accuracy — learned:", learned_accuracy, "random:", random_accuracy)
for row in triple_rows:
    print(row)


In [ ]:
# Try it: inspect the output
example_accuracy, example_rows = triple_accuracy(semantic_index, [("library", "book", "stadium")])
row = example_rows[0]
print(f"library/book: {row['related_cosine']:.3f}")
print(f"library/stadium: {row['unrelated_cosine']:.3f}")
print("Related word scores higher:", bool(example_accuracy))


## Task 18: Visualize the learned geometry

Implement `pca_coordinates` to project a set of vectors onto two principal components using the supplied `principal_directions` helper. Use the supplied plots to inspect training behavior and selected word groups. Consider what the projection loses.


In [ ]:
# Supplied: find the two principal directions; no SVD API knowledge needed.
def principal_directions(centered):
    _, _, directions = torch.linalg.svd(centered, full_matrices=False)
    return directions[:2]

def pca_coordinates(vectors):
    ### Your Solution starts here ###
    pass
    ### Your Solution ends here ###

plot_words = ["library", "book", "school", "teacher", "stadium", "sport", "football", "team",
              "apple", "pear", "fruit", "banana", "cat", "dog", "animal", "horse"]
coordinates = pca_coordinates(game_vectors[[word_to_id[w] for w in plot_words]])
try:
    import matplotlib.pyplot as plt
except ImportError:
    print("Plotting is optional; inspect neighbors and printed metrics instead.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot([r["epoch"] for r in training_history], [r["train_loss"] for r in training_history], label="Training")
    axes[0].plot([r["epoch"] for r in training_history], [r["val_loss"] for r in training_history], label="Validation")
    axes[0].set(xlabel="Epoch", ylabel="Negative-sampling loss")
    axes[0].legend(loc="upper left", fontsize=8)
    ranking_axis = axes[0].twinx()
    ranking_axis.plot([r["epoch"] for r in training_history], [r["pairwise"] for r in training_history],
                      color="green", linestyle="--", label="Validation pairwise")
    ranking_axis.set(ylabel="Validation pairwise", ylim=(0, 1))
    ranking_axis.legend(loc="upper right", fontsize=8)
    axes[1].scatter(coordinates[:, 0], coordinates[:, 1])
    for word, point in zip(plot_words, coordinates):
        axes[1].annotate(word, point.tolist(), xytext=(3, 8 if word == "teacher" else -10 if word == "animal" else 3), textcoords="offset points", fontsize=8)
    axes[1].set(title="Selected words: a 2-D PCA view", xlabel="PC1", ylabel="PC2")
    axes[1].margins(0.15)
    fig.tight_layout()
    plt.show()


In [ ]:
# Try it: inspect the output
print("Selected words in the 2-D projection:")
for word, xy in zip(plot_words[:6], coordinates[:6]):
    print(f"{word:12s} PC1={float(xy[0]): .3f}, PC2={float(xy[1]): .3f}")


## Task 19: Save and reload the model

Implement `vocabulary_hash`, `save_bundle`, and `load_bundle` for a versioned inference checkpoint. Preserve weights, vocabulary order, configuration, source fingerprint, selected epoch, and postprocessing. Validate compatibility when loading and preserve similarities across a round trip.


In [ ]:
def vocabulary_hash(words, context_words):
    ### Your Solution starts here ###
    pass
    ### Your Solution ends here ###

def save_bundle(path, model, words, context_words, config, data_hash, selected_epoch):
    ### Your Solution starts here ###
    pass
    ### Your Solution ends here ###

def load_bundle(path):
    ### Your Solution starts here ###
    pass
    ### Your Solution ends here ###

BUNDLE_PATH = Path("semantic_outputs/dictionary_embeddings.pt")
save_bundle(BUNDLE_PATH, embedding_model, headwords, id_to_context, CONFIG, DATA_HASH, selected_epoch)
loaded_model, loaded_index, bundle_metadata = load_bundle(BUNDLE_PATH)
print("Saved model:", BUNDLE_PATH, "| MiB:", round(BUNDLE_PATH.stat().st_size / 2**20, 1))


In [ ]:
# Try it: inspect the output
print("Saved checkpoint:", BUNDLE_PATH)
print("Selected epoch:", bundle_metadata["selected_epoch"])
print("Neighbors after loading:")
for word, cosine in loaded_index.neighbors("library", 5):
    print(f"  {word:20s} {cosine:.3f}")


### Part IV: build the game

Use learned cosine similarities for feedback and WordNet metadata for clues. The vectors combine word senses, while clues use a fixed sense; ambiguous answers may therefore be confusing. This local notebook exposes Python state, including the answer.


## Task 20: Create progressive clues

Implement `redact_answer`, `build_clues`, and `clue_for` to reveal hints gradually from a fixed WordNet sense: category, increasingly specific word families, then up to two related keywords. Introduce new clues on accepted guesses 1, 3, 5, 7, 9, and 11; keep the current clue between these milestones. Hide words sharing the answer’s Porter stem, avoid synonyms as keywords, and never show the full definition. If evidence runs out, keep the last available clue.


In [ ]:
from nltk.stem import PorterStemmer
stemmer = PorterStemmer()

def redact_answer(text, answer):
    ### Your Solution starts here ###
    pass
    ### Your Solution ends here ###

def build_clues(word):
    ### Your Solution starts here ###
    pass
    ### Your Solution ends here ###

def clue_for(clues, attempt):
    ### Your Solution starts here ###
    pass
    ### Your Solution ends here ###


In [ ]:
# Try it: inspect the output
print("Redaction:", redact_answer("A library with many libraries.", "library"))
for word in ["library", "apple", "river"]:
    print(f"\nClue progression for {word}:")
    example_clues = build_clues(word)
    for attempt in range(1, 13):
        print(f"After guess {attempt:2d}: {clue_for(example_clues, attempt)}")


## Task 21: Score a solution

Implement `solution_points` so only a win earns points: $\lfloor 1000 e^{-t/180}/\sqrt{g} \rfloor$, for elapsed seconds t and accepted guesses g. Reject invalid scoring inputs. Faster solutions with fewer guesses should score higher.


In [ ]:
def solution_points(attempts, elapsed_seconds, won):
    ### Your Solution starts here ###
    pass
    ### Your Solution ends here ###


In [ ]:
# Try it: inspect the output
for attempts, seconds, won in [(1, 10, True), (3, 10, True), (3, 90, True), (12, 90, False)]:
    print(f"{attempts} guesses, {seconds}s, won={won}: {solution_points(attempts, seconds, won)} points")


## Task 22: Describe semantic progress

Implement `semantic_rank`, `temperature`, and `progress`. Rank guesses against the whole vocabulary and compare warmth with the previous accepted guess. Use boiling/hot/warm/cool/cold bands at rank fractions 0.001, 0.01, 0.05, and 0.20; treat cosine differences within 1e-6 as ties.


In [ ]:
def semantic_rank(scores, guess_id, tolerance=1e-6):
    ### Your Solution starts here ###
    pass
    ### Your Solution ends here ###

def temperature(rank, vocabulary_size):
    ### Your Solution starts here ###
    pass
    ### Your Solution ends here ###

def progress(current, previous, tolerance=1e-6):
    ### Your Solution starts here ###
    pass
    ### Your Solution ends here ###


In [ ]:
# Try it: inspect the output
example_scores = semantic_index.scores("library")
previous = None
for word in ["book", "stadium", "library"]:
    guess_id = word_to_id[word]
    score = float(example_scores[guess_id])
    rank = semantic_rank(example_scores, guess_id)
    print(f"{word}: rank={rank}, {temperature(rank, len(headwords))}, {progress(score, previous)}")
    previous = score


## Task 23: Initialize a game round

Implement `start_round` using `RoundState`. A round needs a valid answer, cached target similarities, fixed clues, a positive guess limit, and an injectable monotonic clock. Keep the hidden answer out of the displayed state.


In [ ]:
@dataclass
class RoundState:
    index: SemanticIndex = field(repr=False)
    answer: str = field(repr=False)
    target_scores: torch.Tensor = field(repr=False)
    clues: list = field(repr=False)
    started_at: float
    max_guesses: int
    clock: object = field(repr=False)
    history: list = field(default_factory=list)
    won: bool = False
    over: bool = False
    points: int = 0

def start_round(answer, index, max_guesses=12, clock=time.monotonic):
    ### Your Solution starts here ###
    pass
    ### Your Solution ends here ###


In [ ]:
# Try it: inspect the output
example_round = start_round("library", semantic_index)
print("New round:", example_round)
print("Accepted guesses so far:", len(example_round.history))
print("Vocabulary size:", len(example_round.index.words))


## Task 24: Process a guess

Implement `submit_guess` as a state transition that returns feedback. Invalid, repeated, and post-game guesses must leave the round unchanged. Accepted guesses record similarity, rank, warmth, clue, elapsed time, and terminal status. Exact spelling wins; a win or exhausted guess limit ends the round. Award points once and reveal the answer on a loss. Keep history independent of caller edits.


In [ ]:
def submit_guess(state, text):
    ### Your Solution starts here ###
    pass
    ### Your Solution ends here ###


In [ ]:
# Try it: inspect the output
class TestClock:
    def __init__(self): self.value = 100.0
    def __call__(self): return self.value

example_clock = TestClock()
example_round = start_round("library", semantic_index, clock=example_clock)
for text in ["two words", "book", "BOOK", "stadium", "library"]:
    example_clock.value += 10
    feedback = submit_guess(example_round, text)
    if feedback["accepted"]:
        print(f"{text!r}: {feedback['change']}, attempt {feedback['attempt']}, "
              f"clue={feedback['hint']!r}, points={feedback['points']}")
    else:
        print(f"{text!r}: {feedback['message']}")


## Task 25: Choose suitable hidden answers

Implement `target_pool` and `choose_target` to select reproducibly from a curated set of familiar nouns with embeddings and WordNet clues. Keep the full dictionary available for guesses.


In [ ]:
CURATED_TARGETS = "library mountain river ocean forest desert island village garden kitchen hospital school museum theater stadium airport station bridge bicycle airplane train ship guitar piano violin camera computer telescope microscope book pencil hammer ladder umbrella window mirror bottle basket apple banana potato carrot bread cheese coffee tiger elephant rabbit dolphin butterfly eagle turtle flower mushroom planet comet thunder winter summer morning music language memory dream friendship journey".split()

def target_pool(candidates, index):
    ### Your Solution starts here ###
    pass
    ### Your Solution ends here ###

def choose_target(pool, seed=None):
    ### Your Solution starts here ###
    pass
    ### Your Solution ends here ###

answer_pool = target_pool(CURATED_TARGETS, semantic_index)
print("Accepted guesses:", len(headwords), "| default hidden-answer pool:", len(answer_pool))


In [ ]:
# Try it: inspect the output
print("Some eligible targets:", answer_pool[:12])
print("Repeated selection with seed 9:", [choose_target(answer_pool, 9) for _ in range(3)])
print(f"Players can guess {len(headwords):,} words; default answers come from {len(answer_pool)} targets.")


## Task 26: Make the game playable

Implement `format_feedback` and `play` as a colorful text interface. Show guesses, a warmth bar, similarity, rank, clues, time, and final points in the output cell. Support `:quit`, injectable input/output, and a plain-text option. Keep interactive play opt-in so Run All can finish unattended.


In [ ]:
def color_text(text, code, color=True):
    return f"\033[{code}m{text}\033[0m" if color else str(text)


def format_feedback(result, color=True):
    ### Your Solution starts here ###
    pass
    ### Your Solution ends here ###


def play(index, pool, seed=None, input_fn=input, output_fn=print, color=True):
    ### Your Solution starts here ###
    pass
    ### Your Solution ends here ###


In [ ]:
# Try it: inspect the output
example_inputs = iter(["unknownwordzzz", "book", "library"])
scripted_round = play(semantic_index, ["library"],
                      input_fn=lambda prompt: next(example_inputs))
print("Final state:", scripted_round)


### Play: the word hunt

Uncomment the call below to play. Type your guesses into the notebook’s text prompt and follow the colored feedback in the output cell. Each guess stays visible as part of your trail. Enter `:quit` to stop, or rerun the cell for another round.

Leave `seed=None` for a fresh answer, or use an integer for repeatable rounds. The clock starts when you run the game. Colors use ANSI escape sequences; set `color=False` if your notebook frontend does not render them.


In [ ]:
# Uncomment to start a round.
# round_state = play(semantic_index, answer_pool, seed=None, color=True)


### A complete example round

The answer is fixed to **library** for this demonstration. Feedback uses the learned vectors, including any unintuitive comparisons.


In [ ]:
demo_clock = TestClock()
demo = start_round("library", semantic_index, clock=demo_clock)
for guess in ["book", "stadium", "library"]:
    demo_clock.value += 15
    print(format_feedback(submit_guess(demo, guess)))
    print()


### Part V: evaluate and extend the system

The following tasks investigate weaknesses rather than hiding them: score feedback can reveal more than intended, dictionary-based semantic metrics are limited, spelling similarity differs from meaning, and a single vector merges multiple senses.


## Task 27: Investigate feedback leakage

Implement `consistent_candidates`, `simulate_solver`, and `summarize_sessions` for a solver that knows the embedding index and target pool, but cannot inspect the hidden answer. Filter candidates using cosine feedback rounded to three decimals. Use a fake clock advancing ten seconds per guess and report solve rate, attempts, and synthetic points.


In [ ]:
def consistent_candidates(index, candidates, guess, observed_similarity, decimals=3):
    ### Your Solution starts here ###
    pass
    ### Your Solution ends here ###

def simulate_solver(index, pool, answer, seed=0):
    ### Your Solution starts here ###
    pass
    ### Your Solution ends here ###

def summarize_sessions(sessions):
    ### Your Solution starts here ###
    pass
    ### Your Solution ends here ###

sessions = [simulate_solver(semantic_index, answer_pool, answer, seed=i)
            for i, answer in enumerate(answer_pool[:12])]
print("Known-index score-leakage agent:", summarize_sessions(sessions))


In [ ]:
# Try it: inspect the output
print("Example solver sessions (synthetic timing):")
for answer, session in zip(answer_pool[:3], sessions[:3]):
    print(answer, session)
print("Summary:", summarize_sessions(sessions))


## Task 28: Audit the frozen model

Implement `final_audit` using untouched test edges, fixed negatives sampled with seed 999, and the supplied audit triples. Report coverage, held-out discrimination, semantic accuracy, and target-pool size. Do not tune the model using these results.


In [ ]:
AUDIT_TRIPLES = [("knife", "fork", "galaxy"), ("train", "railway", "cucumber"),
                 ("bird", "eagle", "sofa"), ("rain", "weather", "keyboard"),
                 ("language", "speech", "refrigerator"), ("mountain", "hill", "sandwich")]

def final_audit(model, index):
    ### Your Solution starts here ###
    pass
    ### Your Solution ends here ###

audit = final_audit(embedding_model, semantic_index)
print(json.dumps(audit, indent=2))


In [ ]:
# Try it: inspect the output
print(f"Trained headword coverage: {audit['trained_headword_fraction']:.1%}")
print(f"Held-out pairwise discrimination: {audit['test_pairwise_discrimination']:.1%}")
print(f"Semantic audit accuracy: {audit['audit_triple_accuracy']:.1%}")
print("What does each of these measurements tell you about playing the game?")


**Interpret your audit:** Describe one strength and one uncertainty. Distinguish dictionary coverage from semantic usefulness, and identify what evidence you would need from real players.


## Task 29: Explore character n-grams

Implement `character_buckets` and the unfinished `SubwordEmbeddings` methods. Use bounded character 3–5-grams and stable hashing to share parameters across spellings. Combine a known word’s vector with its mean bucket vector; use buckets alone for unseen spellings. Export composed vectors if you train this extension. Accepted game guesses still follow the dictionary vocabulary.


In [ ]:
def character_buckets(word, bucket_count=20000, min_n=3, max_n=5):
    ### Your Solution starts here ###
    pass
    ### Your Solution ends here ###

class SubwordEmbeddings(DictionaryEmbeddings):
    def __init__(self, words, context_count, dim=96, bucket_count=20000):
        super().__init__(len(words), context_count, dim)
        self.words = list(words)
        self.bucket_count = bucket_count
        self.ngrams = embedding_table(bucket_count, dim, scale=0.05)
        self.word_buckets = []
        for word in words:
            self.word_buckets.append(character_buckets(word, bucket_count))

    def pool_buckets(self, rows):
        device = self.ngrams.weight.device
        vectors = []
        for row in rows:
            ids = torch.tensor(row, dtype=torch.long, device=device)
            vectors.append(self.ngrams(ids).mean(dim=0))
        return torch.stack(vectors)

    def encode_headwords(self, heads):
        ### Your Solution starts here ###
        pass
        ### Your Solution ends here ###

    def forward(self, heads, positives, negatives):
        ### Your Solution starts here ###
        pass
        ### Your Solution ends here ###

    def encode_unseen(self, word):
        ### Your Solution starts here ###
        pass
        ### Your Solution ends here ###

RUN_SUBWORD_TRAINING = False
if RUN_SUBWORD_TRAINING:
    subword_model, subword_history = train_embeddings(
        CONFIG, model_factory=lambda h, c, d: SubwordEmbeddings(headwords, c, d))
    with torch.no_grad():
        composed = torch.cat([subword_model.encode_headwords(torch.arange(i, min(i+1024, len(headwords)), device=DEVICE)).cpu()
                              for i in range(0, len(headwords), 1024)])
    subword_index = SemanticIndex(headwords, composed - composed.mean(dim=0, keepdim=True))
    print("Subword development accuracy:", triple_accuracy(subword_index, DEVELOPMENT_TRIPLES)[0])


In [ ]:
# Try it: inspect the output
example_subwords = SubwordEmbeddings(["library", "librarian"], 2, dim=8, bucket_count=101)
for word in ["library", "librarian", "bookishness"]:
    buckets = character_buckets(word, bucket_count=101)
    print(f"{word}: {len(buckets)} buckets, first IDs={buckets[:8]}")
print("Unseen-spelling vector:", example_subwords.encode_unseen("bookishness").detach().tolist())
print("This extension is untrained; its vectors do not yet demonstrate semantic similarity.")


## Task 30: Explore word senses

Implement `sense_vector` and `compare_senses` using means of learned context vectors for known, non-stop-word definition tokens. Return the five highest-scoring noun sense pairs with their definitions; skip senses without usable vectors. Inspect bank/river and bank/money. This is a diagnostic, not a trained disambiguation model.


In [ ]:
def sense_vector(synset, model, context_to_id):
    ### Your Solution starts here ###
    pass
    ### Your Solution ends here ###

def compare_senses(word_a, word_b, model, context_to_id):
    ### Your Solution starts here ###
    pass
    ### Your Solution ends here ###

for pair in [("bank", "river"), ("bank", "money")]:
    print("Sense comparison:", pair)
    for row in compare_senses(*pair, embedding_model, context_to_id)[:2]:
        print(row)


In [ ]:
# Try it: inspect the output
example_sense = wn.synset("library.n.01")
example_vector = sense_vector(example_sense, embedding_model, context_to_id)
print("Sense:", example_sense.name())
print("Definition:", example_sense.definition())
print("Sense vector (first 8 coordinates):", None if example_vector is None else example_vector[:8].tolist())
print("Best bank/river sense match:")
for match in compare_senses("bank", "river", embedding_model, context_to_id)[:1]:
    print(json.dumps(match, indent=2))



### References

- [Princeton WordNet](https://wordnet.princeton.edu/) and [license](https://wordnet.princeton.edu/license-and-commercial-use); [NLTK WordNet interface](https://www.nltk.org/howto/wordnet.html).
- Mikolov et al. (2013), [Distributed Representations of Words and Phrases and their Compositionality](https://arxiv.org/abs/1310.4546), for negative sampling. Our dictionary-pair adaptation changes the data source.
- Bojanowski et al. (2017), [Enriching Word Vectors with Subword Information](https://aclanthology.org/Q17-1010/), for character n-gram sharing. Task 29 is an instructional implementation, not a reproduction of fastText training.
- PyTorch [Embedding](https://docs.pytorch.org/docs/stable/generated/torch.nn.Embedding.html), [EmbeddingBag](https://docs.pytorch.org/docs/stable/generated/torch.nn.EmbeddingBag.html), and [SparseAdam](https://docs.pytorch.org/docs/stable/generated/torch.optim.SparseAdam.html).
